# Fraud risk analysis
Supplied data; hypothetical cost assumptions. No valid transaction timestamp exists, so results are not temporal validation.

## 1. Environment and reproducibility
Run from the repository root with dependencies in `requirements.txt`. The experiment uses seed 42 plus documented split seeds 43-45.

In [ ]:
%run scripts/run_analysis.py --data data/raw/transaction_fraud_risk_advanced_dataset.csv

In [ ]:
import json, pandas as pd
from IPython.display import display, Image
audit=json.load(open('reports/data_audit.json'))
results=json.load(open('reports/results.json'))
audit

## 2. Data audit and leakage
See the machine-readable audit above. Chargebacks, velocity, distance and upstream scores need proof of availability at authorization time. Transaction IDs and the target are excluded from fitting.

In [ ]:
display(pd.DataFrame({'missing':audit['missing']}))
for i in range(1,4): display(Image(filename=f'reports/figure_{i:02d}.png'))

## 3. Training-only exploratory analysis
Associations are descriptive. Interaction charts have small cells and should not be read causally.

In [ ]:
for i in range(4,13): display(Image(filename=f'reports/figure_{i:02d}.png'))

## 4. Feature engineering and validation
Row-local features are defined in `fraud_pipeline.core.features`. No customer history is reconstructed without customer IDs and timestamps. Stratified train, selection, calibration-fit, policy-selection and test partitions are disjoint.

In [ ]:
display(pd.DataFrame(audit['partitions']).T)

## 5. Baseline, nonlinear models and imbalance
Unweighted and class-weighted logistic, random forest and extra trees models are compared with baseline and engineered features. Selection uses average precision.

In [ ]:
display(pd.read_csv('reports/model_comparison.csv'))
display(pd.read_csv('reports/feature_importance.csv').head(12))

## 6. Calibration
Sigmoid and isotonic calibrators fit on calibration-fit rows and are compared on policy-selection rows. These methods change probability interpretation, while ROC-AUC chiefly measures ranking.

In [ ]:
display(pd.read_csv('reports/calibration_comparison.csv'))
display(Image(filename='reports/figure_14.png'))

## 7. Cost optimization and operations
Costs are assumptions from `configs/cost.json`. Policy search is capacity constrained. The final test simulates outcomes after policy selection.

In [ ]:
display(pd.read_csv('threshold_cost_analysis.csv').groupby('cost_scenario').apply(lambda x:x[x.capacity_ok].nsmallest(1,'total_expected_cost')).reset_index(drop=True))
display(results['test_policy'])

## 8. Explainability and robustness
Permutation importance describes model dependence, not causality. Local reason codes denote threshold routing only. Controlled perturbations retain original labels and do not emulate adaptive fraud.

In [ ]:
display(pd.read_csv('reports/robustness.csv'))
display(pd.read_csv('reports/local_explanations.csv'))

## 9. Final test, monitoring and limitations
Monitor input quality, unknown categories, score and decision rates, mature-label precision and calibration, and review capacity. There is no defensible claim of temporal stability, causal effects, real financial savings, fairness or deployment readiness without new data.

In [ ]:
display(results['test_metrics'])
display(results['approve_all'])